# Extra Lab X3 · A complete dialogue manager for a TV, with the small model

**Time:** 60 min · **Work in teams of 3** · **This is an extra lab:** use it if there is time after Lab 21.

**Where we are in the day:** On Day 2 you saw a dialogue manager for a smart home (Lab 15): the model reads, the graph decides. In Lab 17 you trained a small model to read home commands. This lab puts the two ideas together for a **TV**.
**Today's question:** can a 1.5-billion-parameter model, trained for one minute on generated data, be the "ears" of a complete TV assistant, while plain code does everything else, safely and testably?
**What you will do:**
- Read a LangGraph dialogue manager for a TV and run **ten dialogue patterns** on it (slot filling, context, correction, ambiguity, interruption, confirmation, undo, repair, several commands, a PIN) without a GPU, using keyword rules as the reader.
- Train the small model on generated TV data with the **same recipe as Lab 17**, and put it into the same graph.
- Run about **fifty scenarios** that check every reply and the state of the TV, once with the rules and once with the model.
- Break the manager on purpose, and add one scenario of your own.

**What you will see:** the rules pass every scenario that uses plain, correct English, and fail on noisy speech and Hinglish. The model, with the same graph around it, should pass those too.
**Why it matters on a device:** the model is small and can be wrong, so everything that must be safe (asking before a reset, asking for a PIN, checking a value, undoing) is code that you can test. A new feature is new code and a new scenario, not a new prompt and a hope.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **4 to 5 minutes** in all after the model has downloaded (the training cell is about 1 minute of that). Parts 1 to 3 need **no GPU** and take seconds.
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer.
- **Before you start:** run this lab in its own kernel (Restart Kernel first). Training needs a free GPU, so do not keep the models of other labs loaded.
- **If the training fails** (out of memory, or any error), the lab tells you. If the trainer has put a prepared adapter in `data/tv_adapter_prepared`, the lab uses it by itself. You lose nothing.

**Words used in this lab:**
- **Intent:** what the user wants to do (`set_volume`, `set_channel`, `factory_reset`).
- **Slot:** a detail of the intent (the level, the channel, the app).
- **State:** the notebook of the TV: a plain dictionary (volume, channel, app, power and so on). It is printed after each turn.
- **Task:** what we are in the middle of doing. It is kept between turns.
- **Route:** the decision of the graph about what happens next. The order of its questions is the behaviour.
- **Confirmation, PIN:** two **permission tiers**. A risky action needs a "yes". Adult content needs a PIN.
- **Scenario:** a short conversation with the exact replies we expect and, if we want, the state of the TV at the end.
- **Baseline:** a simple method (here, keyword rules) used to see what the model adds.

**Python in this lab:** loops, `if`, lists and dictionaries and function calls. The code of the manager is in `tv_manager.py`, the TV in `tv_domain.py` and the language reading in `tv_nlu.py`. Open them in a second tab and read while you work.

In [ ]:
import sys, os
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import inspect
import pandas as pd
import tv_domain as td
import tv_nlu as tn
import tv_manager as tm
from slm_lab import check, Timer
from graph_viz import draw_graph
print("scenarios:", len(tm.SCENARIOS), "| TV test messages (golden):", len(tn.load_golden()))
print("adapter found:", tn.find_adapter())

**You should see** the number of scenarios, the number of golden test messages and the adapter that was found. On a first run `adapter found` may be `None` (nothing trained yet) or the prepared adapter. Both are fine for now.

## Part 1: the TV and the manager (no GPU, no waiting)

## X3.1 The state of the TV
Everything the assistant changes is in **one dictionary**. The model never touches it. Only `tv_domain.py` changes it, after the code has checked the request.

**Predict.** How many items does the state have (5, 10 or 17)? Which two would a user never say out loud?

In [ ]:
tv = td.new_tv()
for name in tv:
    print(name, "=", tv[name])
print(len(tv), "items")

**You should see** 17 items: power, volume, muted, input, channel, app, playing and so on. `pin_locked` and `adult_unlocked` belong to the permission tier: the user never says them, the code sets them.

In [ ]:
check(len(tv) >= 15 and tv["volume"] == 20, "Checkpoint: the TV has its state dictionary, volume 20 at the start.",
      "Run the cell of X3.1 again.")

## X3.2 The graph
The manager is a LangGraph graph, built like the home manager of Lab 15. **One node uses the model: `understand`.** Everything else is code. Look at the picture, then at the list of nodes that speak.

In [ ]:
manager = tm.make_manager(use_model=False)       # the reader is a set of keyword rules: no model, no GPU
draw_graph(manager["app"], "The TV dialogue manager")
print("nodes that speak:", tm.SPEAKING)

**You should see** `understand`, then `track`, then a dashed arrow (the decision `route`) to one of the speaking nodes. After a node has spoken, the graph may go back to `track` (the next command in the same sentence) or run `execute` (a task that was just allowed) or stop.

## X3.3 One conversation, with the state printed after each turn
`tm.turn` takes one text and returns the **reply**, the **state** and the **path** (the nodes that ran). `run_conversation` prints all three. We show the whole state dictionary for the first turn only; later we print what changed.

In [ ]:
results = tm.run_conversation(manager, ["turn the volume up"], show="full")
print()
results = tm.run_conversation(manager, ["mute it and go to hdmi 2"], show="changes")

**You should see** the volume going from 20 to 25, then, for the second sentence, the path `understand > track > execute > track > execute`: **one sentence, two commands, two passes** through `track` and `execute`. The sentence was split into two parts and each part was read on its own.

In [ ]:
check(results[0]["state"]["muted"] and results[0]["state"]["input"] == "hdmi 2",
      "Checkpoint: both commands of the sentence were done.", "Run the cells of X3.3 again, from the top of the notebook.")

## X3.4 The decision at the heart: `route`
`route` asks its questions **in order**. The first that fits decides what the assistant does. **The order is the behaviour.** Read it, and find: where does the manager decide that a risky action needs a yes?

In [ ]:
print(inspect.getsource(tm.route))
print(inspect.getsource(tm.route_command))

**Read it:** the questions about what we are waiting for (`awaiting`: a PIN, a yes, a choice) come **first**. Only then does the manager look at a new command, and in the last lines it asks `td.tier(...)`: free, confirm or pin. The model never makes this decision.

## Part 2: the dialogue patterns, one at a time
Each cell sends a short conversation into the **rules** manager. **Predict first:** what will each reply be? The scenarios of Part 3 check these same conversations automatically.

## X3.5 Slot filling, context and correction
The user gives a detail at a time. A **bare number** answers the question that was asked, or **corrects** the last command ("no, 45").

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["set the volume", "40", "no, 45"])             # 🔧 try your own order of details
print()
tm.reset(manager)
_ = tm.run_conversation(manager, ["go to channel 5", "no, channel 7"])

## X3.6 Ambiguity: choices, by number or by name
"A sports channel" fits two channels. The assistant must **offer the choices**, not guess. The user can answer with a number ("the second one") or with a name ("sony").

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["put on a sports channel", "sony"])
print()
tm.reset(manager)
_ = tm.run_conversation(manager, ["play something funny", "the second one", "1"])

**Read it:** the search found three titles, you chose the second, and that title is on **two apps**, so the assistant asked again. Two levels of choice, and the state of the TV changed only at the end.

## X3.7 Interruption, and coming back
In the middle of a task the user asks a question. The assistant answers, **keeps the task** and asks the open question again.

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["set a sleep timer", "what time is it", "30"])

## X3.8 Confirmation (yes, no, cancel) and undo
A risky action waits in `pending` until the user says yes. "No" and "cancel" throw it away. A reset cannot be undone, but a volume change can.

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["factory reset", "no"])
print()
tm.reset(manager)
_ = tm.run_conversation(manager, ["set the volume to 50", "undo", "delete all recordings", "yes", "undo"])

**Read it:** the last "undo" is refused, because `execute` saved no copy of the TV for a deletion. The rule is in the code (`td.IRREVERSIBLE`), so it holds whatever the model says.

## X3.9 A permission tier: the PIN
Adult content needs a PIN. The PIN is read **by code**, in `understand`, so it **never reaches the model**. The list `manager["reads"]` shows exactly what the reader was given.

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["show adult content", "1111", "one two three four"])
print("texts given to the reader:", manager["reads"])

**You should see** the reader was given only `show adult content`. The two PINs, one wrong and one spoken in words, were read by code. Three wrong PINs lock adult content (scenario "three wrong PINs lock it").

In [ ]:
check("1111" not in manager["reads"] and "one two three four" not in manager["reads"],
      "Checkpoint: the PINs were never given to the reader.", "Run the cell of X3.9 again.")

## X3.10 Repair, several commands and out of scope
Not understood: **ask again**, then a **menu**, then **hand over** to a person. A request that is not about the TV is **refused politely** and does not count as a misunderstanding.

In [ ]:
tm.reset(manager)
_ = tm.run_conversation(manager, ["blah blah", "more blah", "hmm"])
print()
tm.reset(manager)
_ = tm.run_conversation(manager, ["set a sleep timer and mute it", "30", "tell me a joke"])

**Read it:** in "set a sleep timer and mute it" the second command **waited** in the queue while the assistant asked for the minutes, and ran as soon as the first was done.

## Part 3: the scenarios (still no GPU)
A scenario is a conversation with the **exact replies** and the **state at the end**, written in `tv_manager.py`. A test that is run by the computer is worth ten that are read by a person.

## X3.11 Run the scenarios with the rules
**Predict.** The scenarios use plain correct English, noisy speech and Hinglish. How many of all the scenarios will the **rules** pass: all, about three quarters, or half? Then run.

In [ ]:
passed_all, total_all, rules_all = tm.run_scenarios(manager)
passed_rules, total_rules, rules_results = tm.run_scenarios(manager, needs="rules")
print("rules, every scenario:", passed_all, "of", total_all)
print("rules, scenarios in plain English:", passed_rules, "of", total_rules)
print("dialogue patterns covered:", tm.patterns_covered())

**You should see** that the rules pass **every scenario in plain English** and fail only those marked for the model (noisy speech, Hinglish, everyday wording). That means the **dialogue logic** (the graph, the state, the tiers) is correct. What is still missing is the **understanding**.

In [ ]:
for result in rules_all:
    if not result["ok"]:
        print("FAIL:", result["name"], "|", result["turns"], "->", result["got"][:1])

In [ ]:
check(passed_rules == total_rules and total_all >= 30, "Checkpoint: the logic passes all the plain-English scenarios, and there are at least 30 scenarios.",
      "Some plain-English scenario failed. Run the cell above again and read the FAIL lines.")

Here is the whole table, one row per scenario.

In [ ]:
pd.DataFrame(tm.scenario_rows(rules_all))

## Part 4: train the small model (GPU, about 1 minute)
The rules cannot read "tern on the tee vee" or "awaaz 40 kar do". The model can learn to. We use the **same recipe as Lab 17**: 4-bit model, LoRA adapter (r = 16), replay data, two epochs, the loss only on the answer. Only the **data** is new: TV commands built from facts.

## X3.12 The training data
Each example is a message and its answer `{"intent": ..., "slots": {...}}`. We build them from facts, in English, in Hinglish and with speech-recognition mistakes. **None of the golden test messages is in the training set.**

In [ ]:
examples = tn.make_training_examples(1300)
print(len(examples), "examples:", tn.count_by_intent(examples))
for example in examples[:6]:
    print(example["text"], "->", tn.make_example(example["text"], example["intent"], example["slots"])["messages"][2]["content"])

In [ ]:
golden = tn.load_golden()
golden_texts = []
for case in golden:
    golden_texts.append(case["text"].lower())
overlap = 0
for example in examples:
    if example["text"].lower() in golden_texts:
        overlap = overlap + 1
check(overlap == 0 and len(examples) >= 1000, "Checkpoint: no golden message is in the training data.", "The training set and the test set overlap.")

## X3.13 Train the adapter
`train_tv_adapter()` is in `tv_nlu.py`. It does what Lab 17 did, step by step, and saves the adapter in `outputs/tv_adapter`. **If it fails for any reason** the cell prints what to do. If the trainer left a prepared adapter, the lab uses it automatically.

In [ ]:
try:
    trained = tn.train_tv_adapter()
    print("trained on", trained["examples"], "examples in", trained["seconds"], "seconds | peak GPU memory", trained["peak_gb"], "GB")
    print("loss:", trained["losses"])
except Exception as problem:
    print("The training did not finish:", problem)
    print("No problem: if data/tv_adapter_prepared exists, the next cell uses it. Otherwise show this message to the trainer.")
ADAPTER = tn.find_adapter()
print("adapter used from here on:", ADAPTER)

**You should see** the loss falling to a small number, a training time of about a minute on a 24 GB GPU, and the path of the adapter that will be used. A tiny loss is not proof (Lab 17): the real test is the golden messages.

In [ ]:
check(ADAPTER is not None, "Checkpoint: there is an adapter to use.",
      "No adapter was found. Run the training cell again, or ask the trainer for data/tv_adapter_prepared.")

## X3.14 Load it and test the reading alone
We load the 1.5-billion-parameter model with the adapter, and test **only the reading** (message to intent and slots) on the golden messages, against the rules. No dialogue yet.

**Predict.** Write your guess for the exact % of the rules and of the model on all golden messages, and the kind of message where the rules are weakest.

In [ ]:
reader = tn.load_reader(ADAPTER)
reader["confidence"] = False                     # we switch the confidence on later; it costs a little time
score_rules = tn.score_reader(tn.make_rules_reader(), golden)
score_model = tn.score_reader(reader, golden)
rows = [{"kind": "ALL", "rules %": score_rules["exact"], "model %": score_model["exact"]}]
for kind in score_model["by_kind"]:
    rows.append({"kind": kind, "rules %": score_rules["by_kind"][kind], "model %": score_model["by_kind"][kind]})
pd.DataFrame(rows)

**You should see** the model above the rules on the whole set, and a big gap on **noisy** and **Hinglish** messages: the rules were written for clean English. Which kinds are still wrong for the model? Look at the list.

In [ ]:
for mistake in score_model["wrong"][:8]:
    print(tn.describe_mistake(mistake))

In [ ]:
check(score_model["exact"] > score_rules["exact"], "Checkpoint: the model reads better than the rules.",
      "The model is not above the rules. Look at the wrong messages above; if the training failed, run X3.13 again.")

## X3.15 How sure is the model?
This is the idea of Lab 20, now for the TV intents: how much probability the model gives to its own intent. Switch it on for three messages.

In [ ]:
reader["confidence"] = True
for text in ["turn the volume up", "tern on the tee vee", "blah blah"]:
    answer = tn.read_text(reader, text)
    print(text, "->", answer["intent"], answer["slots"], "| confidence", answer["confidence"])
reader["confidence"] = False

**You should see** a high confidence for a clear command and a lower one for the nonsense. The manager returns `confidence` in every turn result, so a later lab (X4) can ask back when it is low.

## Part 5: the whole manager with the model

## X3.16 The same scenarios, the same graph, the model as the reader
Only the `understand` node changes. `track`, `route`, the tiers and the tests are the same, so any difference in the results comes from the **understanding**.

**Predict.** How many scenarios will pass with the model: all of them, all but a few, or fewer than the rules?

In [ ]:
manager_model = tm.make_manager(reader=reader)
with Timer() as t_model:
    passed_model, total_model, model_results = tm.run_scenarios(manager_model)
print("model, every scenario:", passed_model, "of", total_model, "in", round(t_model.seconds, 1), "seconds")
for result in model_results:
    if not result["ok"]:
        print("FAIL:", result["name"], "|", result["reason"])

In [ ]:
check(passed_model >= 0.9 * total_model and passed_model > passed_all,
      "Checkpoint: the model passes at least 90 per cent of the scenarios, and more than the rules.",
      "Fewer than 90 per cent pass, or not more than the rules. Read the FAIL lines above: each one is a reading mistake to look at.")

**You should see** the model pass most of the noisy and Hinglish scenarios that the rules fail. **A few misses are normal**: a model trained for one minute is not perfect, and the FAIL lines above are exactly what to learn from (which messages were read wrongly, and what training sentences are missing). If one scenario fails, find out **why** before you do anything: is it a reading mistake (the intent or a slot is wrong) or a logic mistake? Use the next cell.

In [ ]:
table = tm.compare_by_pattern(rules_all, model_results)
pd.DataFrame(table)

**Read it:** the patterns that are pure logic (confirmation, undo, repair, PIN) pass with both readers. The patterns that depend on **how the user speaks** (noisy speech, Hinglish) are where the model earns its place.

In [ ]:
for text in ["awaaz 40 kar do", "turn the volum up by ten", "tv reset kar do"]:
    answer = tn.read_text(reader, text)
    print(text, "->", answer["intent"], answer["slots"], "| raw:", answer["raw"])

## Part 6: break it on purpose

## X3.17 Remove the careful step
Safety is a **decision in code**. To see it, we build a manager whose `route` skips the permission tiers: a risky task goes straight to `execute`. Then we run the scenarios.

**Predict.** Which dialogue patterns will fail, and will the model-based manager be any better?

In [ ]:
def route_no_tier(state):
    chosen = tm.route(state)
    if chosen == "confirm" or chosen == "ask_pin":
        return "execute"                          # 🔧 the careful step is skipped
    return chosen

broken = tm.make_manager(use_model=False, route_fn=route_no_tier)
passed_broken, total_broken, broken_results = tm.run_scenarios(broken, needs="rules")
print("without the tiers:", passed_broken, "of", total_broken, "plain-English scenarios pass")
failed_patterns = []
for result in broken_results:
    if not result["ok"] and result["pattern"] not in failed_patterns:
        failed_patterns.append(result["pattern"])
print("patterns that fail:", failed_patterns)

In [ ]:
check(passed_broken < total_broken and len(broken["errors"]) == 0,
      "Checkpoint: skipping the tiers breaks the confirmation and PIN scenarios, and nothing crashes.",
      "The changed route should break at least one scenario.")

**Read it:** a model that is 100 per cent right on every message would still delete the recordings without asking, because **asking is not the model's job**. That is why the tests check the replies and the state.

## X3.18 The code cleans what the model proposes
The model can return broken JSON, an intent we never defined, or a value outside the range. We feed four bad answers to the reading code, and then a bad value to the TV.

In [ ]:
bad_answers = ['{"intent": "set_volume", "slots": {"level": 500}}', "sure, I will do it", '{"intent": "launch_missiles", "slots": {}}',
               '{"intent": "set_channel", "slots": {"channel": 7, "colour": "red"}}']
for raw in bad_answers:
    answer = tn.read_model_answer(raw)
    print(raw, "->", answer["intent"], answer["slots"], "| dropped:", answer["dropped"], "| valid JSON:", answer["valid_json"])
print("the TV checks the value too:", td.check_value("level", 500))

**You should see** that broken JSON and the unknown intent become `other`, the invented slot `colour` is dropped, and the level 500 is **kept by the reader but refused by the TV** (`check_value`). Two layers: the reader cleans the form, the domain checks the value. Even a perfect model needs the second layer.

In [ ]:
manager_rules = tm.make_manager(use_model=False)
for text in ["", "   ", "asdf qwer " * 30, "!!!"]:
    result = tm.turn(manager_rules, text)
    print(repr(text[:20]), "->", result["reply"][:60])
check(len(manager_rules["errors"]) == 0, "Checkpoint: empty and strange input gives a polite reply and never crashes.",
      "A turn raised an error. Look at manager_rules['errors'].")

## Part 7: your turn

## X3.19 Your turn (about 10 minutes): write a scenario
A scenario is a test you write **before** you trust a feature. Write one of your own: a conversation of two or three turns, the exact replies you expect, and the state at the end. Look at X3.5 to X3.10 for ideas (an undo after a mute, a correction, a question in the middle of a choice). If you are stuck, open the solution cell below the starter.

**Tip:** run the turns first with `tm.run_conversation(manager_rules, [...])`, read the replies, and **decide whether they are right** before you copy them into the scenario. A test copied from output you did not check proves nothing.

In [ ]:
my_scenario = tm.scenario(
    "my scenario", "my pattern",
    ["", ""],                                    # TODO 1: what the user says (two turns)
    ["", ""],                                    # TODO 2: the exact replies you expect
    {},                                          # TODO 3: values the TV must have at the end, for example {"muted": False}
)
my_result = tm.run_scenario(manager_rules, my_scenario)
print("passes with the rules:", my_result["ok"], "|", my_result["reason"])

In [ ]:
check(my_result["ok"] and len(my_scenario["turns"]) >= 2 and my_scenario["turns"][0] != "",
      "Right: your scenario passes. It is now a test that guards this behaviour.",
      "Your scenario does not pass yet. Read the reason above: is the expected reply wrong, or the behaviour?")

Now run the same scenario with the **model** manager. If it passes with the rules and fails with the model, the mistake is in the reading, not in the logic.

In [ ]:
my_result_model = tm.run_scenario(manager_model, my_scenario)
print("passes with the model:", my_result_model["ok"], "|", my_result_model["reason"])

**Go further (if you have time, no code needed):** to add a new intent such as `brightness`, which of the four files would you change, and what test would you write first? Write the list: the TV (`tv_domain.py`), the reader (`tv_nlu.py`: the intent, the slots, training sentences, a rule), the manager (`tv_manager.py`: a required slot, a message) and the scenarios. Then train again with `tn.train_tv_adapter()`.

## What you take away
- The small model **only reads**: message to intent and slots. State, decisions, validation, confirmation, the PIN, undo and repair are code, so they can be tested exactly, on a laptop, with no GPU.
- The **same graph** runs with keyword rules and with the model. The rules show that the logic is right; the model adds understanding of noisy and Hinglish speech. Compare the two on the same scenarios.
- Every behaviour has a **scenario** with exact replies and the final state. When you change something, run all the scenarios.
- Safety is a **decision in code** (`td.tier`, `route`), and values are checked twice (the reader cleans, the TV checks). A PIN never needs to reach a model.

**Good habits from this lab:**
1. Write the scenario first, then the feature.
2. When a scenario fails with the model, find out whether it is a **reading** mistake or a **logic** mistake before you change anything.
3. Keep a test set the model never saw (the golden messages) apart from the scenarios (which test the dialogue).

## Check your understanding
1. In which node of the graph is the model used, and name three nodes that are plain code.
*✍️ Your answer:* 
2. The user says "factory reset" and the model reads it correctly. Which part of the code makes the assistant ask "Shall I go ahead?", and why is the model not trusted with that?
*✍️ Your answer:* 
3. The rules pass every plain-English scenario but the model-based manager passes more scenarios overall. What does that tell you about where the model is needed, and what does it not tell you?
*✍️ Your answer:* 
4. Why is the spoken PIN read by code in `understand` instead of by the model, and how did the lab prove it?
*✍️ Your answer:* 

## Next
**Extra Lab X4:** the assistant gets ears and a voice. Speech goes into Whisper, the text goes into **this manager** (`tm.turn`), and the reply is spoken back, with a timing for each step and a test with noise.